# Diffusion Models from Scratch

We'll build up a diffusion model piece by piece, starting with raw data and ending with sampling.
The goal is to understand every moving part before hiding anything behind a function.

In [ ]:
import math
import torch
import numpy as np
import matplotlib.pyplot as plt
from itertools import pairwise

%matplotlib inline
plt.rcParams['figure.figsize'] = (5, 5)
plt.rcParams['figure.dpi'] = 100

## 1. The Dataset

Let's start with points on a spiral. Nothing fancy — just `t*cos(t)` and `t*sin(t)` scaled down.

In [ ]:
# Raw spiral points
N = 1000
tmin, tmax = np.pi/2, 5*np.pi
t = tmin + torch.linspace(0, 1, N) * tmax
points = torch.stack([t * torch.cos(t) / tmax, t * torch.sin(t) / tmax]).T
points.shape

In [ ]:
plt.scatter(points[:, 0], points[:, 1], s=3, alpha=0.5)
plt.title('Spiral points')
plt.axis('equal');

### What does adding noise look like?

Diffusion works by gradually adding Gaussian noise to data. The amount of noise is controlled by $\sigma$.
The noised version is simply: $x_t = x_0 + \sigma \cdot \epsilon$ where $\epsilon \sim \mathcal{N}(0, I)$.

In [ ]:
# Let's see what pure noise looks like vs noised spiral at different sigma levels
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, sigma in zip(axes, [0.0, 0.05, 0.3, 2.0]):
    eps = torch.randn_like(points)
    noised = points + sigma * eps
    ax.scatter(noised[:, 0], noised[:, 1], s=3, alpha=0.5)
    ax.set_title(f'$\\sigma$ = {sigma}')
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_aspect('equal')

plt.tight_layout()

At $\sigma = 0$ we have the original spiral. As $\sigma$ grows, it becomes pure noise.

### The noise schedule

A schedule is just an increasing list of $\sigma$ values. Log-linear spacing works well for toy problems.

In [ ]:
# Create a log-linear schedule: N sigma values from sigma_min to sigma_max
N_schedule = 200
sigma_min, sigma_max = 0.005, 10.0
sigmas = torch.logspace(math.log10(sigma_min), math.log10(sigma_max), N_schedule)
print(f'Schedule: {len(sigmas)} steps, from {sigmas[0]:.4f} to {sigmas[-1]:.4f}')
plt.plot(sigmas.numpy())
plt.xlabel('Step'); plt.ylabel('$\\sigma$')
plt.title('Log-linear noise schedule');

### Sampling a training batch

During training we pick random $\sigma$ values from the schedule for each example in the batch.
Then we generate random noise $\epsilon$ and compute $x_t = x_0 + \sigma \cdot \epsilon$.

In [ ]:
# Simulate one training batch
batch_size = 64
# Pick random data points (with replacement, as a DataLoader would)
idx = torch.randint(0, len(points), (batch_size,))
x0 = points[idx]

# Pick random sigma for each sample
sigma_idx = torch.randint(0, len(sigmas), (batch_size,))
sigma = sigmas[sigma_idx].unsqueeze(1)  # shape: (batch_size, 1) for broadcasting

# Generate noise and noised data
eps = torch.randn_like(x0)
xt = x0 + sigma * eps

print(f'x0: {x0.shape}, sigma: {sigma.shape}, eps: {eps.shape}, xt: {xt.shape}')

### Wrapping it as a Dataset

Now let's package this into a proper PyTorch dataset — same as `smalldiffusion.Swissroll`.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class Spiral(Dataset):
    def __init__(self, tmin, tmax, N):
        t = tmin + torch.linspace(0, 1, N) * tmax
        self.vals = torch.stack([t * torch.cos(t) / tmax, t * torch.sin(t) / tmax]).T

    def __len__(self): return len(self.vals)
    def __getitem__(self, i): return self.vals[i]

dataset = Spiral(np.pi/2, 5*np.pi, 1000)
loader = DataLoader(dataset, batch_size=2048)
batch = next(iter(loader))
print(f'Batch shape: {batch.shape}')

## 2. The Model

The model's job: given noised data $x_t$ and the noise level $\sigma$, predict the noise $\epsilon$ that was added.

We'll use a simple MLP. The only trick is how we tell the model what $\sigma$ is — we encode it
as $(\sin(0.5 \cdot \log\sigma), \cos(0.5 \cdot \log\sigma))$ and concatenate with the input.

In [ ]:
# Sigma embedding: turn a scalar sigma into a 2D vector the model can use
def get_sigma_embeds(batches, sigma):
    if sigma.shape == torch.Size([]):
        sigma = sigma.unsqueeze(0).repeat(batches)
    s = torch.log(sigma).unsqueeze(1) * 0.5  # (B, 1)
    return torch.cat([torch.sin(s), torch.cos(s)], dim=1)  # (B, 2)

# Quick test
sig = torch.tensor([0.1, 1.0, 5.0])
get_sigma_embeds(3, sig)

In [ ]:
from torch import nn

# A minimal MLP that takes (x, sigma) -> predicted noise
# Input: 2D point + 2D sigma embed = 4 dims
class NoisePredMLP(nn.Module):
    def __init__(self, hidden_dims=(16, 128, 128, 128, 128, 16)):
        super().__init__()
        layers = []
        in_dim = 4  # 2 for data + 2 for sigma embedding
        for out_dim in hidden_dims:
            layers.extend([nn.Linear(in_dim, out_dim), nn.GELU()])
            in_dim = out_dim
        layers.append(nn.Linear(in_dim, 2))  # output same dim as data
        self.net = nn.Sequential(*layers)

    def forward(self, x, sigma):
        sigma_embeds = get_sigma_embeds(x.shape[0], sigma.squeeze())  # (B, 2)
        nn_input = torch.cat([x, sigma_embeds], dim=1)                # (B, 4)
        return self.net(nn_input)

model = NoisePredMLP()
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

### One forward pass (before any training)

Let's make sure the shapes work. The model sees noised data and a sigma, and should output
something the same shape as the input (its guess at the noise).

In [ ]:
x0 = batch[:8]
sigma = sigmas[torch.randint(0, len(sigmas), (8,))].unsqueeze(1)  # (8, 1)
eps = torch.randn_like(x0)
xt = x0 + sigma * eps

with torch.no_grad():
    eps_pred = model(xt, sigma)

print(f'Input xt:     {xt.shape}')
print(f'True noise:   {eps.shape}')
print(f'Predicted:    {eps_pred.shape}')
print(f'Loss (MSE):   {nn.MSELoss()(eps, eps_pred).item():.4f}  (random, untrained)')

## 3. Training

The training loop is dead simple:
1. Sample a batch of data $x_0$
2. Pick random $\sigma$ from the schedule
3. Sample noise $\epsilon \sim \mathcal{N}(0, I)$
4. Compute noised input $x_t = x_0 + \sigma \cdot \epsilon$
5. Model predicts $\hat{\epsilon}(x_t, \sigma)$
6. Loss = MSE between $\epsilon$ and $\hat{\epsilon}$

Let's write it out explicitly.

In [ ]:
from tqdm import tqdm

model = NoisePredMLP()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
epochs = 15000
losses = []

for epoch in (pbar := tqdm(range(epochs))):
    for x0 in loader:
        model.train()
        optimizer.zero_grad()

        # Sample random sigmas for this batch
        sigma = sigmas[torch.randint(0, len(sigmas), (x0.shape[0],))].unsqueeze(1)

        # Add noise
        eps = torch.randn_like(x0)
        xt = x0 + sigma * eps

        # Predict noise and compute loss
        eps_pred = model(xt, sigma)
        loss = nn.MSELoss()(eps, eps_pred)

        loss.backward()
        optimizer.step()
        losses.append(loss.item())

plt.plot(losses[::100])  # plot every 100th to keep it readable
plt.xlabel('Step (x100)'); plt.ylabel('Loss')
plt.title('Training loss');

## 4. Inference (Sampling)

Sampling reverses the noising process. We start from pure noise $x_T \sim \mathcal{N}(0, \sigma_{\max}^2 I)$
and iteratively denoise. The simplest sampler (DDIM-style) is:

$$x_{t-1} = x_t - (\sigma_t - \sigma_{t-1}) \cdot \hat{\epsilon}(x_t, \sigma_t)$$

That's it — subtract a fraction of the predicted noise at each step.

### Subsampling the schedule for inference

We trained with 200 sigma values but we don't need all of them for sampling.
We subsample to get a smaller, decreasing sequence.

In [ ]:
# Subsample the schedule: take `steps` evenly-spaced values, in decreasing order
# "trailing" spacing as in Table 2 of https://arxiv.org/abs/2305.08891
def sample_sigmas(sigmas, steps):
    N = len(sigmas)
    indices = list((N * (1 - np.arange(0, steps)/steps)).round().astype(np.int64) - 1)
    return sigmas[indices + [0]]  # includes the final sigma=sigma_min

sample_sigs = sample_sigmas(sigmas, 20)
print(f'{len(sample_sigs)} sigma values (20 steps + final):')
print(sample_sigs)

### Explicit sampling loop

Using the accelerated sampler from the paper (`gam=2`), which averages the current
and previous noise predictions for better denoising.

In [ ]:
@torch.no_grad()
def sample(model, sigmas, n_samples=1000, gam=2.0):
    """Returns list of xt at every step, for visualization."""
    xt = torch.randn(n_samples, 2) * sigmas[0]  # start from noise
    trajectory = [xt.clone()]
    eps_prev = None

    for i, (sig, sig_prev) in enumerate(pairwise(sigmas)):
        eps = model(xt, sig.expand(n_samples))
        # Average with previous prediction (gam=2 means double weight on current)
        eps_av = eps * gam + eps_prev * (1 - gam) if i > 0 else eps
        eps_prev = eps
        xt = xt - (sig - sig_prev) * eps_av
        trajectory.append(xt.clone())

    return trajectory

In [ ]:
sample_sigs = sample_sigmas(sigmas, 20)
trajectory = sample(model, sample_sigs, n_samples=1000)

# Show every 4th step + the final result
steps_to_show = list(range(0, len(trajectory), 4)) + [len(trajectory) - 1]
steps_to_show = sorted(set(steps_to_show))

fig, axes = plt.subplots(1, len(steps_to_show), figsize=(4 * len(steps_to_show), 4))
for ax, step in zip(axes, steps_to_show):
    pts = trajectory[step].numpy()
    ax.scatter(pts[:, 0], pts[:, 1], s=3, alpha=0.5)
    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal')
    label = 'Pure noise' if step == 0 else f'Step {step}' if step < len(trajectory)-1 else 'Final'
    ax.set_title(label)
plt.suptitle('Denoising trajectory', y=1.02, fontsize=14)
plt.tight_layout()

In [ ]:
# Final samples vs original data
final = trajectory[-1].numpy()
orig = dataset.vals.numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].scatter(orig[:, 0], orig[:, 1], s=3, alpha=0.5)
axes[0].set_title('Original data'); axes[0].set_aspect('equal')
axes[1].scatter(final[:, 0], final[:, 1], s=3, alpha=0.5, c='C1')
axes[1].set_title('Generated samples'); axes[1].set_aspect('equal')
for ax in axes: ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
plt.tight_layout()

## 5. Diffusion Samples, Not Inverts

A critical insight: the diffusion model does **not** try to invert the noise to find the
exact original position of each point. Instead, it learns the data distribution and **samples
plausible starting points** that could have produced the noisy input.

To demonstrate this, let's:
1. Pick 10 specific noise inputs and color them red
2. Run two independent sampling processes from the **same starting noise**
3. Show that the 10 points land in **different places** each run — and different from any original point

In [ ]:
# Create a fixed batch of starting noise
n_samples = 500
n_special = 10

# Use a fixed seed for the starting noise so both runs share it
torch.manual_seed(42)
xt_init = torch.randn(n_samples, 2) * sample_sigs[0]

# The first 10 points are our "special" tracked points
special_start = xt_init[:n_special].clone()

In [ ]:
@torch.no_grad()
def sample_from_init(model, sigmas, xt_init, gam=2.0):
    """Sample starting from a specific initial noise, with fresh randomness in the model."""
    xt = xt_init.clone()
    trajectory = [xt.clone()]
    eps_prev = None

    for i, (sig, sig_prev) in enumerate(pairwise(sigmas)):
        eps = model(xt, sig.expand(xt.shape[0]))
        eps_av = eps * gam + eps_prev * (1 - gam) if i > 0 else eps
        eps_prev = eps
        xt = xt - (sig - sig_prev) * eps_av
        trajectory.append(xt.clone())

    return trajectory

Wait — DDIM sampling with `gam=2, mu=0` is actually **deterministic** given the same starting noise
(no extra noise is injected at each step). So two runs from the same `xt_init` will give identical
results. That's a feature of DDIM, not a bug!

To show that diffusion **samples** rather than **inverts**, we instead need to show that:
1. The same starting noise, denoised deterministically, lands on **plausible but arbitrary** spiral positions
2. **Different** starting noise lands on **different** plausible positions

So we'll use two different noise initializations for our 10 tracked points and compare where they end up.

In [ ]:
# Run 1: fixed seed for initial noise
torch.manual_seed(42)
xt_run1 = torch.randn(n_samples, 2) * sample_sigs[0]
traj_run1 = sample_from_init(model, sample_sigs, xt_run1)

# Run 2: different seed for initial noise
torch.manual_seed(123)
xt_run2 = torch.randn(n_samples, 2) * sample_sigs[0]
traj_run2 = sample_from_init(model, sample_sigs, xt_run2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Original data with 10 highlighted original points for reference
ax = axes[0]
orig = dataset.vals.numpy()
ax.scatter(orig[:, 0], orig[:, 1], s=3, alpha=0.3, c='gray', label='Original data')
# Mark first 10 dataset points in green as reference
ax.scatter(orig[:n_special, 0], orig[:n_special, 1], s=60, c='green', marker='*',
           zorder=5, edgecolors='black', linewidths=0.5, label=f'{n_special} reference points')
ax.set_title('Original data')
ax.legend(fontsize=8, loc='lower right')

# Run 1
ax = axes[1]
final1 = traj_run1[-1].numpy()
ax.scatter(final1[n_special:, 0], final1[n_special:, 1], s=3, alpha=0.3, c='gray')
ax.scatter(final1[:n_special, 0], final1[:n_special, 1], s=60, c='red', marker='*',
           zorder=5, edgecolors='black', linewidths=0.5, label=f'{n_special} tracked')
ax.set_title('Run 1 (seed=42)')
ax.legend(fontsize=8, loc='lower right')

# Run 2
ax = axes[2]
final2 = traj_run2[-1].numpy()
ax.scatter(final2[n_special:, 0], final2[n_special:, 1], s=3, alpha=0.3, c='gray')
ax.scatter(final2[:n_special, 0], final2[:n_special, 1], s=60, c='red', marker='*',
           zorder=5, edgecolors='black', linewidths=0.5, label=f'{n_special} tracked')
ax.set_title('Run 2 (seed=123)')
ax.legend(fontsize=8, loc='lower right')

for ax in axes:
    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal')

plt.suptitle('Diffusion samples plausible positions, not fixed inversions', y=1.02, fontsize=13)
plt.tight_layout()

### Zooming in: tracking the 10 points through the denoising process

Let's watch where the 10 tracked points travel during denoising in both runs.

In [ ]:
steps_to_show = [0, 3, 6, 10, 15, len(traj_run1)-1]

fig, axes = plt.subplots(2, len(steps_to_show), figsize=(4 * len(steps_to_show), 8))

for row, (traj, seed) in enumerate([(traj_run1, 42), (traj_run2, 123)]):
    for col, step in enumerate(steps_to_show):
        ax = axes[row, col]
        pts = traj[step].numpy()
        ax.scatter(pts[n_special:, 0], pts[n_special:, 1], s=3, alpha=0.2, c='gray')
        # Color each of the 10 special points with a distinct color
        colors = plt.cm.tab10(np.arange(n_special))
        ax.scatter(pts[:n_special, 0], pts[:n_special, 1], s=60, c=colors, marker='*',
                   zorder=5, edgecolors='black', linewidths=0.5)
        ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
        ax.set_aspect('equal')
        if row == 0:
            label = 'Pure noise' if step == 0 else f'Step {step}' if step < len(traj)-1 else 'Final'
            ax.set_title(label)
        if col == 0:
            ax.set_ylabel(f'Run (seed={seed})', fontsize=12)

plt.suptitle('Same 10 point indices end up in different places each run', y=1.01, fontsize=13)
plt.tight_layout()

In [ ]:
# Quantify: how far apart are the 10 points between the two runs?
final1_special = traj_run1[-1][:n_special]
final2_special = traj_run2[-1][:n_special]
dists = (final1_special - final2_special).norm(dim=1)
print(f'Distances between the {n_special} tracked points across runs:')
for i, d in enumerate(dists):
    print(f'  Point {i}: {d.item():.4f}')
print(f'\nMean distance: {dists.mean().item():.4f}')
print(f'\nThis confirms: same index, different noise -> different final position.')
print('The model learned the distribution, not a mapping from noise to specific points.')